# Global Mean

Predict the training mean for every pair.


## Load splits


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from utils.evaluation import mean_squared_error, concordance_index, hit_at_k

PAIR_KEYS = ["compound_iso_smiles", "target_sequence"]
SEED = 42
splits = {
    name: {part: pd.read_csv(Path("data/splits") / f"{name}_{part}.csv",
                           float_precision="round_trip")
           for part in ["train", "val", "test"]}
    for name in ["davis", "kiba"]
}


## Predictions and MSE

Check saved models and calculate MSE.


In [2]:
import json

MODEL_DIR = Path("saved_models")

SCORE_SCALES = {
    "davis": {"name": "pKd", "description": "Supplied pKd values; no additional log transformation."},
    "kiba": {"name": "KIBA score", "description": "Original supplied KIBA scale; no transformation."},
}


def fit_global_mean(training_data, dataset_name):
    # Training mean
    return {
        "model": "Global Mean",
        "dataset": dataset_name,
        "training_mean": float(training_data["affinity"].mean()),
        "training_observations": len(training_data),
        "score_scale": SCORE_SCALES[dataset_name],
        "random_seed": SEED,
    }


def predict_global_mean(model, pairs):
    # Constant prediction
    return np.full(len(pairs), model["training_mean"], dtype=float)


result_rows = []
for dataset_name, partitions in splits.items():
    model = fit_global_mean(partitions["train"], dataset_name)
    assert model["training_mean"] == float(partitions["train"]["affinity"].mean())
    model_path = MODEL_DIR / f"{dataset_name}_global_mean.json"
    restored_model = json.loads(model_path.read_text())
    assert restored_model == model
    row = {
        "Dataset": dataset_name.upper(),
        "Score scale": model["score_scale"]["name"],
        "Training mean": model["training_mean"],
    }
    for partition_name, part in partitions.items():
        pairs = part[PAIR_KEYS]
        predictions = predict_global_mean(model, pairs)
        np.testing.assert_array_equal(predictions, predict_global_mean(restored_model, pairs))
        assert np.isfinite(predictions).all()
        row[f"{partition_name.capitalize()} MSE"] = mean_squared_error(part["affinity"], predictions)
    # Check training variance
    np.testing.assert_allclose(row["Train MSE"], partitions["train"]["affinity"].var(ddof=0))
    result_rows.append(row)

results = pd.DataFrame(result_rows)
print(results.to_string(index=False, float_format=lambda value: f"{value:.8f}"))
print("Checkpoint predictions match.")


Dataset Score scale  Training mean  Train MSE    Val MSE   Test MSE
  DAVIS         pKd     5.39147631 0.64982728 0.67511968 0.67995842
   KIBA  KIBA score    11.72100643 0.69332749 0.68780518 0.71770140
Checkpoint predictions match.


## CI and Hit@K

CI: prediction ties score 0.5; equal actual scores are excluded. No comparable pairs gives NaN.

Hit@K: check for a true-best target in the top K per ligand. Break prediction ties by target ID. Include all observed candidates if fewer than K; singletons always hit.

Global Mean has CI = 0.5. Its Hit@K reflects tie-breaking, not personalised ranking.


In [3]:
extended_results = results.copy()
for row_index, dataset_name in enumerate(["davis", "kiba"]):
    model = json.loads((MODEL_DIR / f"{dataset_name}_global_mean.json").read_text())
    for part_name, part in splits[dataset_name].items():
        predictions = predict_global_mean(model, part[PAIR_KEYS])
        ci = concordance_index(part["affinity"], predictions)
        assert ci == 0.5
        extended_results.loc[row_index, f"{part_name.capitalize()} CI"] = ci
        if part_name == "train":
            continue
        for k in [1, 5]:
            hit = hit_at_k(part, predictions, k)
            # Check tie-breaking
            shuffled = part.sample(frac=1, random_state=42).reset_index(drop=True)
            assert hit == hit_at_k(shuffled, predict_global_mean(model, shuffled[PAIR_KEYS]), k)
            extended_results.loc[row_index, f"{part_name.capitalize()} Hit@{k}"] = hit["hit_rate"]
            extended_results.loc[row_index, f"{part_name.capitalize()} eligible ligands"] = hit["eligible_ligands"]

print(extended_results.to_string(index=False, float_format=lambda value: f"{value:.8f}"))


Dataset Score scale  Training mean  Train MSE    Val MSE   Test MSE   Train CI     Val CI  Val Hit@1  Val eligible ligands  Val Hit@5    Test CI  Test Hit@1  Test eligible ligands  Test Hit@5
  DAVIS         pKd     5.39147631 0.64982728 0.67511968 0.67995842 0.50000000 0.50000000 0.10294118           68.00000000 0.13235294 0.50000000  0.11764706            68.00000000  0.17647059
   KIBA  KIBA score    11.72100643 0.69332749 0.68780518 0.71770140 0.50000000 0.50000000 0.37569903         1967.00000000 0.71784443 0.50000000  0.37519221          1951.00000000  0.70835469


## Results

DAVIS: test MSE = 0.6800, Hit@1 = 0.1176, Hit@5 = 0.1765.

KIBA: test MSE = 0.7177, Hit@1 = 0.3752, Hit@5 = 0.7084.

CI is 0.5 for both datasets across all splits. Hit@K depends on target-ID tie-breaking because all predictions are equal.
